# جلسهٔ ۰۳ — مسائل ارضای محدودیت (CSP)

**سطح متوسط | قابل اجرا روی CPU | آفلاین | بدون نیاز به کلید API**

در مسئلهٔ ارضای محدودیت (Constraint Satisfaction Problem)، باید به مجموعه‌ای از متغیرها مقدار بدهیم، به‌گونه‌ای که همهٔ محدودیت‌های تعریف‌شده رعایت شوند. بازگشت به عقب (Backtracking) و انتخاب هوشمندانهٔ ترتیب متغیرها، این روش را برای کارهایی مانند زمان‌بندی امتحان‌ها و رنگ‌آمیزی نقشه کاربردی می‌کنند.

### اهداف یادگیری
- مفهوم **متغیرها (Variables)** و **دامنهٔ مقادیر (Domains)** را توضیح دهید و برای آن‌ها مثال بزنید.
- **سازگاری (Consistency)** و **بازگشت به عقب (Backtracking)** را در حل مسئله به کار ببرید.
- روش انتخاب متغیر با **کمترین مقادیر باقی‌مانده (MRV)** را توضیح دهید و اثر آن را بر ترتیب جست‌وجو بررسی کنید.
- تفاوت **یافتن پاسخ معتبر (Feasibility)** و **یافتن پاسخ بهینه (Optimization)** را تشخیص دهید.

ابتدا **README.md** اصلی دوره را بخوانید. پیش از اجرای هر سلول، نتیجهٔ آن را حدس بزنید.

## مثال حل‌شده

می‌خواهیم ناحیه‌های یک نقشهٔ کوچک را با سه رنگ رنگ‌آمیزی کنیم، به‌طوری که دو ناحیهٔ همسایه هم‌رنگ نباشند.

پس از رنگ‌کردن یک ناحیه، رنگ‌های ناسازگار را از گزینه‌های ناحیه‌های مجاور حذف کنید. سپس مثالی نشان دهید که در آن یک انتخاب اولیه به بن‌بست می‌رسد و باید با **بازگشت به عقب** انتخاب قبلی تغییر کند.

**پرسش کلاسی:** سه جزء اصلی یک مسئلهٔ ارضای محدودیت (CSP) چیست؟

## مدل رنگ‌آمیزی نقشه

دو ناحیهٔ همسایه نباید رنگ یکسانی داشته باشند.

- `ADJ` همسایگی ناحیه‌ها را مشخص می‌کند (نام‌ها مخفف ایالت‌های استرالیا هستند).
- `COLORS` رنگ‌های مجاز را تعریف می‌کند.
- تابع `available` رنگ‌هایی را برمی‌گرداند که با رنگ ناحیه‌های همسایهٔ قبلاً مقداردهی‌شده تداخل ندارند.

در این مرحله هنوز همهٔ ناحیه‌ها رنگ نشده‌اند؛ فقط دامنهٔ مجاز یک ناحیه را بررسی می‌کنیم.

In [ ]:
ADJ={"WA":["NT","SA"],"NT":["WA","SA","Q"],"SA":["WA","NT","Q","NSW","V"],"Q":["NT","SA","NSW"],"NSW":["SA","Q","V"],"V":["SA","NSW"],"T":[]}
COLORS=("red","green","blue")
def available(v,a):
    return [c for c in COLORS if all(a.get(n)!=c for n in ADJ[v])]
print("دامنهٔ اولیه:",available("WA",{}))

## بازگشت به عقب و روش MRV

الگوریتم `solve` رنگ‌ها را به‌ترتیب امتحان می‌کند و وقتی راه‌حل معتبری پیدا نشود، انتخاب قبلی را برمی‌گرداند.

با فعال بودن `mrv=True`، در هر مرحله متغیری انتخاب می‌شود که **کمترین تعداد رنگ مجاز باقی‌مانده** را دارد. در حالت `mrv=False`، متغیرها به‌ترتیب اولیه بررسی می‌شوند.

برای چنین نمونهٔ کوچکی، **تعداد فراخوانی‌های بازگشتی** معیار مناسب‌تری از زمان اجرا برای مقایسهٔ این دو روش است. به یاد داشته باشید که MRV یک راهبرد برای ترتیب جست‌وجو است؛ محدودیت تازه‌ای به مسئله اضافه نمی‌کند.

In [ ]:
def solve(mrv=True):
    calls=[0]
    def search(a):
        calls[0]+=1
        if len(a)==len(ADJ):return dict(a)
        rest=[v for v in ADJ if v not in a]
        v=min(rest,key=lambda x:len(available(x,a))) if mrv else rest[0]
        for color in available(v,a):
            a[v]=color;result=search(a)
            if result is not None:return result
            del a[v]
        return None
    return search({}),calls[0]
assignment,steps=solve()
print(assignment,"فراخوانی‌ها با MRV:",steps,"فراخوانی‌ها با ترتیب ثابت:",solve(False)[1])
assert all(assignment[v]!=assignment[n] for v in ADJ for n in ADJ[v])

## تمرین

رنگ‌های مجاز یکی از ناحیه‌ها را محدود کنید و دوباره تعداد فراخوانی‌های بازگشتی را بشمارید. نتایج حالت `mrv=True` و `mrv=False` را مقایسه کنید.

**تکلیف:** یک مسئلهٔ زمان‌بندی برای سه امتحان طراحی کنید که در آن محدودیت‌هایی مانند تداخل اتاق یا زمان برگزاری وجود دارد.

**برداشت نادرست رایج:** معتبر بودن یک مقداردهیِ جزئی به این معنی نیست که حتماً می‌توان آن را به یک راه‌حل کامل تبدیل کرد.

نوت‌بوک را از یک کرنل تازه و به‌ترتیب اجرا کنید و دست‌کم یکی از محدودیت‌های روش خود را توضیح دهید.